pdf 파일 다운받아 사용 
- 2040_seoul_plan
- OneNYC_2050_Strategic_Plan

In [15]:
# poetry add langchain_huggingface

from glob import glob

for g in glob('./*.pdf'):
    print(g)

.\2040_seoul_plan.pdf
.\OneNYC_2050_Strategic_Plan.pdf
.\unsu.pdf


In [16]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import os

In [17]:
# pdf를 읽어서 chunk로 쪼개서 반환
def read_pdf_and_split_text(pdf_path, chunk_size=1000, chunk_overlap=100):
    """
    주어진 PDF 파일을 읽고 텍스트를 분할합니다.
    매개변수:
        pdf_path (str): PDF 파일의 경로.
        chunk_size (int, 선택적): 각 텍스트 청크의 크기. 기본값은 1000입니다.
        chunk_overlap (int, 선택적): 청크 간의 중첩 크기. 기본값은 100입니다.
    반환값:
        list: 분할된 텍스트 청크의 리스트.
    """
    print(f'PDF : {pdf_path} ------------')

    pdf_loader = PyPDFLoader(pdf_path) # pdf 읽을 객체 pdf_loader
    data_from_pdf = pdf_loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    )

    splits = text_splitter.split_documents(data_from_pdf)

    print(f'Number of splits: {len(splits)}\n') # split 개수 확인
    return splits

In [18]:
# poetry add sentence-transformers

embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device':'cpu'}, # cuda 지원하면 cuda, 아닌 경우 cpu
    encode_kwargs={'normalize_embeddings':True}, # 임베딩 정규화
)

In [19]:
embeddings.embed_documents('안녕') # 임베딩 -> 수치로 결과값 출력

[[0.002456849440932274,
  0.03226263448596001,
  -0.007424331270158291,
  0.005268442444503307,
  -0.05809727683663368,
  -0.03042869083583355,
  -0.00830000638961792,
  0.03674247860908508,
  0.009622770361602306,
  -0.007633928209543228,
  0.017204441130161285,
  0.03887706995010376,
  -0.029649006202816963,
  -0.021820027381181717,
  -0.0008699532481841743,
  -0.032562535256147385,
  0.03178010880947113,
  -0.02246292680501938,
  0.023502610623836517,
  -0.012333753518760204,
  -0.03881017863750458,
  -0.017907725647091866,
  0.03872939199209213,
  0.01371566392481327,
  0.025562576949596405,
  0.022995982319116592,
  -0.027591736987233162,
  0.027073420584201813,
  -0.009775801561772823,
  -0.026509815827012062,
  -0.006837829481810331,
  -0.02930246852338314,
  0.028823168948292732,
  -0.07535600662231445,
  -0.03274376690387726,
  -0.004341269377619028,
  -0.023116104304790497,
  0.021545160561800003,
  -0.054298121482133865,
  0.05281256511807442,
  0.04333125799894333,
  -0.020

In [ ]:
persist_directory = './chroma_store/' # 이 폴더에 데이터 저장

if os.path.exists(persist_directory): # 폴더에 존재한다면
    print('Loading existing Chroma store')
    vectorstore = Chroma( # 존재하는 경우라서 -> .from_documents 사용 X
        persist_directory=persist_directory,
        embedding_function=embeddings
    )
else:
    print('Creating new Chroma store')
    all_splits = []
    for g in glob('./*.pdf'):
        all_splits.extend(read_pdf_and_split_text(g))
    
    print(f'Total number of splits : {len(all_splits)}')
    
    vectorstore = Chroma.from_documents(
        documents=all_splits,
        embedding=embeddings,
        persist_directory=persist_directory
    )

Creating new Chroma store
PDF : .\2040_seoul_plan.pdf ------------
Number of splits: 308

PDF : .\OneNYC_2050_Strategic_Plan.pdf ------------
Number of splits: 1023

PDF : .\unsu.pdf ------------
Number of splits: 15

Total number of splits : 1346


TypeError: Chroma.__init__() got an unexpected keyword argument 'embeddings'

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={'k':5}) # 키워드 5개 
chunks = retriever.invoke("서울시 쓰레기 저감 정책")

for chunks in chunks:
    print(chunk.metadata)
    print(chunk.page_content)